# indicator_tokenizer — Colab Notebook

Fits quantile-based boundaries for 6 technical indicators
(RSI, MACD histogram, Bollinger %B, ATR, Volume Ratio, Price-vs-SMA) on
training-period candles, then saves the `.npy` boundaries that the other
projects (`late_fusion_agent`, `multimodal_encoder`, `moe_trading_agent`)
depend on.

**How to run**: `Runtime` → GPU not required (CPU is fine) → `Run all`.

By default it generates synthetic candles (so it works out of the box); for
real data set `USE_MOCK_DATA = False` and mount your Drive.


In [ ]:
!pip install -q pyarrow polars pyyaml scikit-learn pandas
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())


## 1. Indicator computer


In [ ]:
"""Technical-indicator computer (from indicator_tokenizer/indicators/computer.py)."""
import numpy as np


def _ema(arr, span):
    alpha = 2.0 / (span + 1)
    out = np.zeros(len(arr), dtype=np.float64)
    out[0] = arr[0]
    for i in range(1, len(arr)):
        out[i] = alpha * arr[i] + (1 - alpha) * out[i - 1]
    return out


class IndicatorComputer:
    def rsi(self, close, period=14):
        delta = np.diff(close.astype(np.float64), prepend=close[0])
        gain = np.where(delta > 0, delta, 0.0)
        loss = np.where(delta < 0, -delta, 0.0)
        avg_gain = _ema(gain, period)
        avg_loss = _ema(loss, period)
        rs = avg_gain / (avg_loss + 1e-10)
        return (100 - 100 / (1 + rs)).astype(np.float32)

    def macd_hist(self, close, fast=12, slow=26, signal=9):
        c = close.astype(np.float64)
        macd_line = _ema(c, fast) - _ema(c, slow)
        signal_line = _ema(macd_line, signal)
        return (macd_line - signal_line).astype(np.float32)

    def bollinger_pctb(self, close, period=20, num_std=2.0):
        c = close.astype(np.float64)
        out = np.zeros(len(c), dtype=np.float32)
        for i in range(period - 1, len(c)):
            w = c[i - period + 1 : i + 1]
            sma = w.mean(); std = w.std()
            upper = sma + num_std * std; lower = sma - num_std * std
            out[i] = (c[i] - lower) / (upper - lower + 1e-10)
        return out

    def atr(self, high, low, close, period=14):
        tr = np.zeros(len(close), dtype=np.float64)
        tr[0] = high[0] - low[0]
        tr[1:] = np.maximum(
            high[1:] - low[1:],
            np.maximum(
                np.abs(high[1:].astype(np.float64) - close[:-1].astype(np.float64)),
                np.abs(low[1:].astype(np.float64) - close[:-1].astype(np.float64)),
            ),
        )
        return _ema(tr, period).astype(np.float32)

    def volume_ratio(self, close, volume, period=20):
        v = volume.astype(np.float64)
        out = np.zeros(len(v), dtype=np.float32)
        for i in range(period - 1, len(v)):
            sma = v[i - period + 1 : i + 1].mean()
            out[i] = v[i] / (sma + 1e-10)
        return out

    def price_vs_sma(self, close, period=20):
        c = close.astype(np.float64)
        out = np.zeros(len(c), dtype=np.float32)
        for i in range(period - 1, len(c)):
            sma = c[i - period + 1 : i + 1].mean()
            out[i] = (c[i] - sma) / (sma + 1e-10)
        return out

    def compute_all(self, ohlcv):
        return {
            "rsi": self.rsi(ohlcv["close"]),
            "macd_hist": self.macd_hist(ohlcv["close"]),
            "bollinger_pctb": self.bollinger_pctb(ohlcv["close"]),
            "atr": self.atr(ohlcv["high"], ohlcv["low"], ohlcv["close"]),
            "volume_ratio": self.volume_ratio(ohlcv["close"], ohlcv["volume"]),
            "price_vs_sma": self.price_vs_sma(ohlcv["close"]),
        }


## 2. Indicator tokenizer


In [ ]:
"""Indicator tokenizer (from indicator_tokenizer/indicators/tokenizer.py)."""
from pathlib import Path
import numpy as np


class FixedBoundaries:
    def __init__(self, bins, offset=2):
        self.bins = np.array(bins, dtype=np.float32)
        self.offset = offset
        self.vocab_size = len(bins) + 1 + offset

    def encode_batch(self, values):
        return (np.searchsorted(self.bins, values, side="right") + self.offset).astype(np.int32)

    def save(self, path): np.save(path, self.bins)
    def load(self, path): self.bins = np.load(path)


class QuantileBoundaries:
    def __init__(self, n_bins, offset=2):
        self.n_bins = n_bins
        self.offset = offset
        self.vocab_size = n_bins + offset
        self.boundaries = None

    def fit(self, values):
        q = np.linspace(0, 100, self.n_bins + 1)[1:-1]
        self.boundaries = np.percentile(values, q).astype(np.float32)

    def encode_batch(self, values):
        assert self.boundaries is not None
        return (np.searchsorted(self.boundaries, values, side="right") + self.offset).astype(np.int32)

    def save(self, path): np.save(path, self.boundaries)
    def load(self, path): self.boundaries = np.load(path)


class IndicatorTokenizer:
    PAD_ID = 0; SPECIAL_ID = 1

    def __init__(self):
        self.rsi = FixedBoundaries(bins=[20, 30, 70, 80])
        self.macd_hist = QuantileBoundaries(n_bins=7)
        self.bollinger_pctb = FixedBoundaries(bins=[0.0, 0.25, 0.75, 1.0])
        self.atr = QuantileBoundaries(n_bins=6)
        self.volume_ratio = QuantileBoundaries(n_bins=5)
        self.price_vs_sma = QuantileBoundaries(n_bins=5)
        self._quantile_fields = ["macd_hist", "atr", "volume_ratio", "price_vs_sma"]
        self._all_fields = ["rsi", "macd_hist", "bollinger_pctb", "atr",
                            "volume_ratio", "price_vs_sma"]

    def fit(self, ind):
        for f in self._quantile_fields:
            getattr(self, f).fit(ind[f])

    def encode(self, ind):
        return {f: getattr(self, f).encode_batch(ind[f]) for f in self._all_fields}

    def vocab_sizes(self):
        return {f: getattr(self, f).vocab_size for f in self._all_fields}

    def save(self, d):
        d = Path(d); d.mkdir(parents=True, exist_ok=True)
        for f in self._all_fields:
            getattr(self, f).save(d / f"{f}.npy")

    def load(self, d):
        d = Path(d)
        for f in self._all_fields:
            getattr(self, f).load(d / f"{f}.npy")


## 3. Configuration


In [ ]:
"""Configuration."""
from pathlib import Path

USE_MOCK_DATA = True
if USE_MOCK_DATA:
    DATA_DIR = Path("/content/mock_data")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/trading_data")

MOCK_MONTHS = ["2024-01", "2024-02", "2024-03"]
MOCK_MINUTES_PER_MONTH = 20000
BOUNDARIES_DIR = Path("/content/boundaries")
SYMBOL = "BTCUSDT"
TRAIN_MONTHS = ("2024-01", "2024-03")

if Path("/content/drive/MyDrive").exists():
    ARTIFACTS_ROOT = Path("/content/drive/MyDrive/w_training/indicator_tokenizer")
else:
    ARTIFACTS_ROOT = Path("/content/artifacts/indicator_tokenizer")
print("DATA_DIR:", DATA_DIR, "| BOUNDARIES_DIR:", BOUNDARIES_DIR)
print("ARTIFACTS_ROOT:", ARTIFACTS_ROOT)


## 4. Mock data generation


In [ ]:
"""Generate synthetic BTCUSDT 1-min OHLCV (geometric Brownian motion).
Skipped automatically when USE_MOCK_DATA = False."""
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq


def _gen_month(n_minutes, start_price, seed):
    rng = np.random.default_rng(seed)
    sigma = 0.0008
    lr = rng.normal(0.0, sigma, size=n_minutes)
    closes = start_price * np.exp(np.cumsum(lr))
    opens = np.concatenate([[start_price], closes[:-1]])
    noise = np.abs(rng.normal(0, sigma, size=n_minutes)) * closes
    highs = np.maximum(opens, closes) + noise
    lows = np.minimum(opens, closes) - noise
    vols = rng.lognormal(3.0, 1.0, size=n_minutes).astype(np.float32)
    return {
        "open": opens.astype(np.float32),
        "high": highs.astype(np.float32),
        "low": lows.astype(np.float32),
        "close": closes.astype(np.float32),
        "volume": vols,
    }


if USE_MOCK_DATA:
    kd = DATA_DIR / "BTCUSDT" / "klines_1m"
    kd.mkdir(parents=True, exist_ok=True)
    start_price = 42000.0
    for i, m in enumerate(MOCK_MONTHS):
        out = kd / f"{m}.parquet"
        if out.exists():
            print("skip", out.name); continue
        d = _gen_month(MOCK_MINUTES_PER_MONTH, start_price, 42 + i)
        start_price = float(d["close"][-1])
        pq.write_table(pa.table(d), out)
        print(f"wrote {out.name}  rows={MOCK_MINUTES_PER_MONTH}")
    print("files:", sorted(p.name for p in kd.glob("*.parquet")))
else:
    print("USE_MOCK_DATA=False — expecting real data at", DATA_DIR)


## 5. Fit boundaries on training data


In [ ]:
"""Fit indicator boundaries on training months and save them."""
import pyarrow.parquet as pq
import numpy as np

klines_dir = DATA_DIR / SYMBOL / "klines_1m"
start, end = TRAIN_MONTHS
files = sorted([f for f in klines_dir.glob("*.parquet") if start <= f.stem <= end])
print(f"loading {len(files)} months of training data")

all_ind = {k: [] for k in ["rsi","macd_hist","bollinger_pctb","atr","volume_ratio","price_vs_sma"]}
comp = IndicatorComputer()
for f in files:
    t = pq.read_table(f)
    ohlcv = {c: np.array([v.as_py() for v in t.column(c)], dtype=np.float32)
             for c in ["open","high","low","close","volume"]}
    ind = comp.compute_all(ohlcv)
    for k in all_ind:
        all_ind[k].append(ind[k])
combined = {k: np.concatenate(v) for k, v in all_ind.items()}
for k, v in combined.items():
    print(f"  {k:20s} n={len(v):>8,} min={v.min():8.4f} max={v.max():8.4f} mean={v.mean():8.4f} std={v.std():8.4f}")

tok = IndicatorTokenizer()
tok.fit(combined)
tok.save(BOUNDARIES_DIR)
print(f"\nSaved boundaries to {BOUNDARIES_DIR}")
for k, v in tok.vocab_sizes().items():
    print(f"  {k:20s} vocab_size={v}")


## 6. Verify: reload & encode


In [ ]:
"""Verify: reload saved boundaries, encode a sample, show token histograms."""
import numpy as np
from collections import Counter

reloaded = IndicatorTokenizer()
reloaded.load(BOUNDARIES_DIR)
print("reloaded vocab sizes:", reloaded.vocab_sizes())

encoded = reloaded.encode(combined)
for k, arr in encoded.items():
    counts = Counter(arr.tolist())
    top = sorted(counts.items())
    print(f"{k:20s} token distribution: {top}")


## 7. Save artifacts to Google Drive
Copies fitted boundaries and config into `ARTIFACTS_ROOT` (Google Drive when mounted, `/content/artifacts/...` otherwise).


In [ ]:
"""Save artifacts (fitted boundaries + config) to ARTIFACTS_ROOT."""
import json
import shutil
from datetime import datetime
from pathlib import Path

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = ARTIFACTS_ROOT / f"run_{RUN_TAG}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

dst = RUN_DIR / "boundaries"
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(BOUNDARIES_DIR, dst)

with open(RUN_DIR / "config.json", "w") as f:
    json.dump({
        "symbol": SYMBOL,
        "train_months": list(TRAIN_MONTHS),
        "vocab_sizes": tok.vocab_sizes(),
        "data_dir": str(DATA_DIR),
    }, f, indent=2)

print(f"saved to: {RUN_DIR}")
for p in sorted(RUN_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(RUN_DIR)!s:40s}  {p.stat().st_size:>10,} B")
